# 🎬 YouTube → Українське дублювання

**Повний пайплайн:**
YouTube URL → завантаження → транскрипція (Whisper) → переклад → озвучення (TTS) → готове відео

---
### ⚡ Перед запуском:
1. `Runtime → Change runtime type → T4 GPU`
2. Запускай клітинки по порядку
3. Остання клітинка відкриє Gradio інтерфейс

## 📦 Крок 1 — Встановлення залежностей
*(~3-5 хвилин, запускається один раз)*

In [1]:
# Системні залежності
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ ffmpeg встановлено')

# Python пакети
!pip install -q faster-whisper
print('✅ faster-whisper встановлено')

!pip install -q deep-translator
print('✅ deep-translator встановлено')

# edge-tts замість Coqui TTS (підтримує Python 3.12+, має Ukrainian голоси)
!pip install -q edge-tts
print('✅ edge-tts встановлено')

!pip install -q yt-dlp
print('✅ yt-dlp встановлено')

!pip install -q moviepy
print('✅ moviepy встановлено')

!pip install -q gradio
print('✅ gradio встановлено')

!pip install -q pydub
print('✅ pydub встановлено')

print('\n🎉 Всі залежності встановлено!')

✅ ffmpeg встановлено
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 93.3 MB/s eta 0:00:00
✅ faster-whisper встановлено
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.1 MB/s eta 0:00:00
✅ deep-translator встановлено
✅ edge-tts встановлено
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.1 MB/s eta 0:00:00
✅ yt-dlp встановлено
✅ moviepy встановлено
✅ gradio встановлено
✅ pydub встановлено

🎉 Всі залежності встановлено!


## 🛠️ Крок 2 — Імпорти та налаштування

In [2]:
import os
import json
import asyncio
import subprocess
import shutil
import torch
import gradio as gr
from pathlib import Path
from pydub import AudioSegment
from deep_translator import GoogleTranslator
from faster_whisper import WhisperModel
import edge_tts

# Перевірка GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Пристрій: {device}')
if device == 'cuda':
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU не знайдено — буде повільніше')

# Робоча директорія
WORK_DIR = Path('/content/dubbing_workspace')
WORK_DIR.mkdir(exist_ok=True)
print(f'📁 Робоча директорія: {WORK_DIR}')

🖥️ Пристрій: cuda
🎮 GPU: Tesla T4
📁 Робоча директорія: /content/dubbing_workspace


## 🧠 Крок 3 — Завантаження моделей
*(~2-3 хвилини, завантажується один раз)*

In [3]:
# --- Whisper (транскрипція) ---
print('⏳ Завантажую Whisper large-v3...')
compute = 'float16' if device == 'cuda' else 'int8'
whisper_model = WhisperModel('large-v3', device=device, compute_type=compute)
print('✅ Whisper готовий')

# --- edge-tts (синтез мовлення) ---
# Доступні Ukrainian голоси:
#   uk-UA-PolinaNeural  — жіночий
#   uk-UA-OstapNeural   — чоловічий
TTS_VOICE = 'uk-UA-PolinaNeural'
print(f'✅ edge-tts готовий (голос: {TTS_VOICE})')

print('\n🎉 Всі моделі завантажені!')

⏳ Завантажую Whisper large-v3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Whisper готовий
✅ edge-tts готовий (голос: uk-UA-PolinaNeural)

🎉 Всі моделі завантажені!


## ⚙️ Крок 4 — Функції пайплайну

In [6]:
def download_video(url: str, out_dir: Path) -> tuple[Path, Path]:
    """Завантажує відео та аудіо з YouTube."""
    video_path = out_dir / 'input_video.mp4'
    audio_path = out_dir / 'input_audio.wav'

    print(f'⬇️ Завантажую відео: {url}')
    subprocess.run([
        'yt-dlp',
        '-f', 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        '--merge-output-format', 'mp4',
        '-o', str(video_path),
        '--no-playlist',
        url
    ], check=True)

    # Витягуємо аудіо
    print('🔊 Витягую аудіо...')
    subprocess.run([
        'ffmpeg', '-y', '-i', str(video_path),
        '-ar', '22050', '-ac', '1',
        str(audio_path)
    ], check=True, capture_output=True)

    print('✅ Відео завантажено')
    return video_path, audio_path


def transcribe(audio_path: Path) -> list[dict]:
    """Транскрибує аудіо за допомогою Whisper."""
    print('📝 Транскрибую аудіо...')
    segments, info = whisper_model.transcribe(
        str(audio_path),
        language='en',
        beam_size=5,
        vad_filter=True  # фільтрує тишу
    )
    chunks = [{
        'start': s.start,
        'end': s.end,
        'text': s.text.strip()
    } for s in segments if s.text.strip()]

    print(f'✅ Транскрибовано {len(chunks)} фрагментів')
    return chunks


def translate_chunks(chunks: list[dict]) -> list[dict]:
    """Перекладає фрагменти з англійської на українську."""
    print('🌐 Перекладаю на українську...')
    translator = GoogleTranslator(source='en', target='uk')

    for i, chunk in enumerate(chunks):
        try:
            chunk['ua_text'] = translator.translate(chunk['text'])
        except Exception as e:
            print(f'⚠️ Помилка перекладу фрагменту {i}: {e}')
            chunk['ua_text'] = chunk['text']  # залишаємо оригінал

    print('✅ Переклад завершено')
    return chunks


async def _synthesize_chunk(text: str, out_file: Path, voice: str):
    """Асинхронно синтезує один фрагмент через edge-tts."""
    communicate = edge_tts.Communicate(text, voice)
    mp3_file = out_file.with_suffix('.mp3')
    await communicate.save(str(mp3_file))
    # Конвертуємо MP3 → WAV для pydub
    audio = AudioSegment.from_mp3(str(mp3_file))
    audio.export(str(out_file), format='wav')
    mp3_file.unlink(missing_ok=True)


def synthesize_speech(chunks: list[dict], out_dir: Path, voice: str = TTS_VOICE) -> list[Path]:
    """Синтезує українське мовлення через edge-tts."""
    print('🎙️ Синтезую українське мовлення...')
    chunks_dir = out_dir / 'chunks'
    chunks_dir.mkdir(exist_ok=True)
    audio_files = []

    for i, chunk in enumerate(chunks):
        out_file = chunks_dir / f'{i:04d}.wav'
        try:
            asyncio.run(_synthesize_chunk(chunk['ua_text'], out_file, voice))
            audio_files.append(out_file)
        except Exception as e:
            print(f'⚠️ Помилка синтезу фрагменту {i}: {e}')
            silence = AudioSegment.silent(duration=int((chunk['end'] - chunk['start']) * 1000))
            silence.export(out_file, format='wav')
            audio_files.append(out_file)

        if (i + 1) % 10 == 0:
            print(f'   {i + 1}/{len(chunks)} фрагментів...')

    print('✅ Синтез завершено')
    return audio_files


def stretch_audio(audio: AudioSegment, target_duration_ms: int) -> AudioSegment:
    """Розтягує або стискає аудіо під потрібну тривалість за допомогою ffmpeg atempo."""
    current_duration = len(audio)
    if current_duration == 0:
        return AudioSegment.silent(duration=target_duration_ms)

    ratio = current_duration / target_duration_ms
    # ffmpeg atempo filter rate must be between 0.5 and 2.0
    atempo_rate = max(0.5, min(2.0, ratio))

    if abs(atempo_rate - 1.0) < 0.01: # If close to 1.0, no need to process
        return audio

    temp_input_wav = Path('/tmp/temp_stretch_input.wav')
    temp_output_wav = Path('/tmp/temp_stretch_output.wav')

    audio.export(str(temp_input_wav), format='wav')

    try:
        subprocess.run([
            'ffmpeg', '-y',
            '-i', str(temp_input_wav),
            '-filter:a', f'atempo={atempo_rate}',
            '-vn', # no video
            str(temp_output_wav)
        ], check=True, capture_output=True)

        stretched_audio = AudioSegment.from_wav(str(temp_output_wav))
        return stretched_audio
    except subprocess.CalledProcessError as e:
        print(f"Error during ffmpeg time stretching: {e.stderr.decode()}")
        # Fallback to silent or original audio in case of error
        return AudioSegment.silent(duration=target_duration_ms)
    finally:
        temp_input_wav.unlink(missing_ok=True)
        temp_output_wav.unlink(missing_ok=True)


def assemble_audio(chunks: list[dict], audio_files: list[Path], out_dir: Path) -> Path:
    """Збирає фінальне аудіо з таймкодами."""
    print('🔧 Збираю фінальне аудіо...')
    total_ms = int(chunks[-1]['end'] * 1000) + 2000
    final_audio = AudioSegment.silent(duration=total_ms)

    for i, (chunk, audio_file) in enumerate(zip(chunks, audio_files)):
        part = AudioSegment.from_wav(str(audio_file))
        target_duration = int((chunk['end'] - chunk['start']) * 1000)
        part = stretch_audio(part, target_duration)
        position_ms = int(chunk['start'] * 1000)
        final_audio = final_audio.overlay(part, position=position_ms)

    out_path = out_dir / 'ukrainian_audio.wav'
    final_audio.export(str(out_path), format='wav')
    print('✅ Аудіо зібрано')
    return out_path


def merge_video_audio(video_path: Path, audio_path: Path, out_dir: Path) -> Path:
    """Замінює аудіо у відео на українське."""
    print('🎬 Збираю фінальне відео...')
    out_path = out_dir / 'output_ukrainian.mp4'
    subprocess.run([
        'ffmpeg', '-y',
        '-i', str(video_path),
        '-i', str(audio_path),
        '-map', '0:v:0',
        '-map', '1:a:0',
        '-c:v', 'copy',
        '-shortest',
        str(out_path)
    ], check=True, capture_output=True)
    print('✅ Відео готове!')
    return out_path


def save_transcript(chunks: list[dict], out_dir: Path):
    """Зберігає транскрипт та переклад у SRT та TXT."""
    def fmt_time(t):
        h, r = divmod(t, 3600)
        m, s = divmod(r, 60)
        ms = int((s % 1) * 1000)
        return f'{int(h):02}:{int(m):02}:{int(s):02},{ms:03}'

    srt_path = out_dir / 'subtitles_ua.srt'
    with open(srt_path, 'w', encoding='utf-8') as f:
        for i, chunk in enumerate(chunks, 1):
            f.write(f"{i}\n")
            f.write(f"{fmt_time(chunk['start'])} --> {fmt_time(chunk['end'])}\n")
            f.write(f"{chunk.get('ua_text', chunk['text'])}\n\n")

    txt_path = out_dir / 'transcript.txt'
    with open(txt_path, 'w', encoding='utf-8') as f:
        for chunk in chunks:
            f.write(f"[{chunk['start']:.1f}s] EN: {chunk['text']}\n")
            f.write(f"[{chunk['start']:.1f}s] UA: {chunk.get('ua_text', '')}\n\n")

    return srt_path, txt_path


print('✅ Всі функції завантажено!')

✅ Всі функції завантажено!


## 🎨 Крок 5 — Gradio інтерфейс

In [5]:
def run_dubbing(youtube_url: str, voice_choice: str, progress=gr.Progress()):
    """Головна функція — повний пайплайн дублювання."""

    if not youtube_url.strip():
        return None, None, None, '❌ Введіть YouTube посилання'

    voice = 'uk-UA-PolinaNeural' if voice_choice == 'Polina (жіночий)' else 'uk-UA-OstapNeural'

    # Очищаємо робочу директорію
    job_dir = WORK_DIR / 'current_job'
    if job_dir.exists():
        shutil.rmtree(job_dir)
    job_dir.mkdir()

    try:
        # 1. Завантаження
        progress(0.1, desc='⬇️ Завантажую відео з YouTube...')
        video_path, audio_path = download_video(youtube_url.strip(), job_dir)

        # 2. Транскрипція
        progress(0.25, desc='📝 Транскрибую (Whisper)...')
        chunks = transcribe(audio_path)

        if not chunks:
            return None, None, None, '❌ Не вдалося розпізнати мовлення'

        # 3. Переклад
        progress(0.45, desc='🌐 Перекладаю на українську...')
        chunks = translate_chunks(chunks)

        # 4. Синтез мовлення
        progress(0.60, desc='🎙️ Синтезую українське мовлення...')
        audio_files = synthesize_speech(chunks, job_dir, voice)

        # 5. Збірка аудіо
        progress(0.80, desc='🔧 Збираю аудіо з таймкодами...')
        final_audio = assemble_audio(chunks, audio_files, job_dir)

        # 6. Збірка відео
        progress(0.90, desc='🎬 Збираю фінальне відео...')
        final_video = merge_video_audio(video_path, final_audio, job_dir)

        # 7. Зберігаємо субтитри
        progress(0.95, desc='📄 Зберігаю субтитри...')
        srt_path, txt_path = save_transcript(chunks, job_dir)

        progress(1.0, desc='✅ Готово!')

        report = f"""✅ Дублювання завершено!

📊 Статистика:
• Фрагментів: {len(chunks)}
• Тривалість: {chunks[-1]['end']:.1f} сек
• Голос: {voice}

📝 Перші 3 фрагменти:
"""
        for chunk in chunks[:3]:
            report += f"EN: {chunk['text']}\nUA: {chunk.get('ua_text', '')}\n\n"

        return str(final_video), str(srt_path), str(txt_path), report

    except Exception as e:
        import traceback
        error_msg = f'❌ Помилка: {str(e)}\n\n{traceback.format_exc()}'
        print(error_msg)
        return None, None, None, error_msg


# --- Gradio UI ---
with gr.Blocks(title='🎬 YouTube UA Dubbing', theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # 🎬 YouTube → Українське дублювання
    **Автоматичний переклад та озвучення відео з англійської на українську**

    Пайплайн: `Whisper (транскрипція)` → `Google Translate` → `edge-tts (синтез голосу)`
    """)

    with gr.Row():
        with gr.Column(scale=2):
            url_input = gr.Textbox(
                label='🔗 YouTube посилання',
                placeholder='https://www.youtube.com/watch?v=...',
                lines=1
            )
            voice_input = gr.Radio(
                choices=['Polina (жіночий)', 'Ostap (чоловічий)'],
                value='Polina (жіночий)',
                label='🎤 Голос'
            )
            run_btn = gr.Button('🚀 Запустити дублювання', variant='primary', size='lg')

            gr.Markdown("""
            **⏱️ Орієнтовний час обробки (T4 GPU):**
            - 10 хв відео → ~15-20 хв
            - 30 хв відео → ~40-50 хв
            - 60 хв відео → ~90-120 хв
            """)

        with gr.Column(scale=3):
            status_box = gr.Textbox(
                label='📊 Статус та звіт',
                lines=12,
                interactive=False
            )

    gr.Markdown('### 📥 Результати')

    with gr.Row():
        video_out = gr.Video(label='🎬 Відео українською')

    with gr.Row():
        srt_out = gr.File(label='📄 Субтитри (.srt)')
        txt_out = gr.File(label='📝 Транскрипт (.txt)')

    run_btn.click(
        fn=run_dubbing,
        inputs=[url_input, voice_input],
        outputs=[video_out, srt_out, txt_out, status_box]
    )

    gr.Markdown("""
    ---
    **⚠️ Примітки:**
    - edge-tts використовує хмарний синтез Microsoft (потрібен інтернет)
    - Lip-sync не включено (для цього потрібен Wav2Lip — окремий крок)
    - Colab безкоштовний сеанс: до 12 годин
    """)

# Запускаємо з публічним посиланням
app.launch(share=True, debug=False)

/tmp/ipykernel_1017/3913021871.py:71: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='🎬 YouTube UA Dubbing', theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5fccf9c3d986020c4c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
